#1 IMPORT THE DATA

In [6]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Sarves Srirangan\kaggle\PS_20174392719_1491204439457_log.csv")

In [8]:
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [9]:
df.shape

(6362620, 11)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [11]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [12]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

#2 DATA CLEANING

In [13]:
# Check missing values
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [15]:
# Check duplicates
int(df.duplicated().sum())

0

In [16]:
# Check transaction types
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [18]:
# Check negative or zero amounts
int((df['amount'] <= 0).sum())

16

In [20]:
# Check if balances make logical sense
int((df['oldbalanceOrg'] - df['amount'] != df['newbalanceOrig']).sum())

5413997

In [21]:
df.groupby('type')['isFraud'].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

#3 FEATURE ENGINEERING

In [24]:
#1 Transaction Type Encoding
df['type_encoded'] = df['type'].map({
    'PAYMENT': 0,
    'TRANSFER': 1,
    'CASH_OUT': 2,
    'CASH_IN': 3,
    'DEBIT': 4
})

In [25]:
#2 Balance Difference
df['balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']

In [26]:
#3 Amount to Balance Ratio
df['amount_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

In [27]:
#4 High Value Transaction
df['is_high_value'] = df['amount'].apply(lambda x: 1 if x > 200000 else 0)

In [28]:
#5 Transaction Count per User
df['transaction_count'] = df.groupby('nameOrig')['nameOrig'].transform('count')

In [29]:
#6 Merchant Transaction Flag
df['is_merchant'] = df['nameDest'].apply(lambda x: 1 if x.startswith('M') else 0)

In [33]:
#7 Balance Error Flag
df['balance_error'] = (
    (abs((df['oldbalanceOrg'] - df['amount']) - df['newbalanceOrig']) > 100) |
    (abs((df['oldbalanceDest'] + df['amount']) - df['newbalanceDest']) > 100)
).astype(int)

In [36]:
#8 Risky Transaction Type
df['is_risky_type'] = df['type'].apply(lambda x: 1 if x in ['TRANSFER', 'CASH_OUT'] else 0)

In [37]:
#9 Zero Balance After Transaction
df['is_zero_balance'] = df['newbalanceOrig'].apply(lambda x: 1 if x == 0 else 0)

In [38]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_encoded,balance_diff,amount_ratio,is_high_value,transaction_count,is_merchant,balance_error,is_risky_type,is_zero_balance
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,0,9839.64,0.057834,0,1,1,1,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,0,1864.28,0.087731,0,1,1,1,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,1,181.00,0.994505,0,1,0,1,1,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,2,181.00,0.994505,0,1,0,1,1,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,0,11668.14,0.280788,0,1,1,1,0,0


#4 FRAUD DETECTION LOGIC

In [40]:
# Rule-Based Fraud Indicators
df['rule_high_amount'] = (df['amount'] > 200000).astype(int)
df['rule_balance_error'] = df['balance_error']
df['rule_risky_type'] = df['is_risky_type']
df['rule_high_ratio'] = (df['amount_ratio'] > 0.9).astype(int)
df['rule_high_frequency'] = (df['transaction_count'] > 10).astype(int)

In [41]:
#Risk Score Calculation
df['risk_score'] = (
    df['rule_high_amount'] * 30 +
    df['rule_balance_error'] * 25 +
    df['rule_risky_type'] * 20 +
    df['rule_high_ratio'] * 15 +
    df['rule_high_frequency'] * 10
)

In [42]:
# Risk Classification
def classify_risk(score):
    if score >= 60:
        return 'High Risk'
    elif score >= 30:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['risk_level'] = df['risk_score'].apply(classify_risk)

In [44]:
# Compare risk level with actual fraud
pd.crosstab(df['risk_level'], df['isFraud'])

isFraud,0,1
risk_level,,
High Risk,2708422,6891
Low Risk,1914659,16
Medium Risk,1731326,1306


In [43]:
df[['amount','type','risk_score','risk_level','isFraud']].head(10)

,amount,type,risk_score,risk_level,isFraud
0,9839.64,PAYMENT,25,Low Risk,0
1,1864.28,PAYMENT,25,Low Risk,0
2,181.00,TRANSFER,60,High Risk,1
3,181.00,CASH_OUT,60,High Risk,1
4,11668.14,PAYMENT,25,Low Risk,0
5,7817.71,PAYMENT,25,Low Risk,0
6,7107.77,PAYMENT,25,Low Risk,0
7,7861.64,PAYMENT,25,Low Risk,0
8,4024.36,PAYMENT,40,Medium Risk,0
9,5337.77,DEBIT,25,Low Risk,0


#5 EXPORT THE DATA

In [ ]:
df.to_csv(r"C:\Users\Sarves Srirangan\OneDrive\Desktop\data_analysis_projects\fraud_detection_analytics_1\fraud_final.csv", index=False)